## ML1 - Machine Learning Activity 1 - Data Collection
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: May 15th, 2026

### About: In this notebook you'll start the machine leraning process by collecting data. This is the first step in the machine learning process. 

<img src="Graphics/ML_Process.png" width="1000" height="250">


In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# At the start of a python program we often import code libraries to add to our programs functions
# Importing the Sphero RVR Tank Robot Software Development Kit
from sphero_sdk import SpheroRvrObserver
from sphero_sdk import RawMotorModesEnum
from sphero_sdk import DriveFlagsBitmask
import time
# Managing event loops in python/jupyter
import nest_asyncio
nest_asyncio.apply()

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# Prepare the interface for the robot and wake up the robot so I'll be ready to receive commands
rvr = SpheroRvrObserver()
time.sleep(2)
rvr.wake()

## Machine Learning for Object Detection and Avoidance
## Our goal is to creat an AI that will pilot our robot.
## We want this AI to avoid hitting obstacles and stay within the robot play area. So we have to show it where to drive and what an obstacle is!
## <span style="color:orange">Without us, it wouldn't know!</span>

## We teach our AI using photos, collecting lots of photos is critical for machine learning.
## Today we'll collect two types of photos: 
## "**Blocked**" photos or data shows the robot what it looks like when there is an object in front of it or when it is at the edge of the play area.
## "**Free**" photos or data shows the robot what it looks like when the robot is free to drive forward, an hopefully not hit anything!

<img src="Graphics/Blocked_Example.png" width="1000" height="250">


<img src="Graphics/Free_Example.png" width="1000" height="250">


## **Data Collection:**
## In this notebook we'll move the robot and collect images for our AI to learn from.


## To start this process, we're now going to collect data with our robots.

### Since we want our AI to identify obstacles in front of the robot, we have to take lots of pictures.
### If our robot's path is blocked by some object, examples: chair, backpack, shoe, wall, then we'll take a photo and save it in the "blocked" directory. If the robot's path is free of obstacles, we'll take a photo and save it to the "free" directory.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# Run this cell to start the camera and display live video

# Important libraries for using the camera
import ipywidgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jetcam.csi_camera import CSICamera # This represents instructions for making a camera
from uuid import uuid1
import os
import ipywidgets.widgets as widgets # This library is useful for creating a graphical user interface


# Import new library to help us draw a line
import cv2 # Open computer vision library (good for images and video)

# Instantiate (Create) the camera object - Object Oriented Programming (OOP)
camera = CSICamera(width=224, height=224) # Make a new object called "camera"
image = camera.read()
camera.running = True

# Creating a Graphical User Interface (GUI) for displaying the image
image_widget = ipywidgets.Image(format='jpeg')

# Callback function for each time a new frame is collected
def update_image(change):
    image = change['new']
    # Process the frame and display it
    image_widget.value = bgr8_to_jpeg(image)

# Start the update event observation and display the GUI
camera.observe(update_image, names='value')
display(image_widget)

### Now that we have a the camera running, let's draw a useful line on the image to help us define how close an object should be for it to be considered an obstacle.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# NEXT We want to draw a line on the camera display
# Creating a GUI for displaying the image
image_widget_2 = ipywidgets.Image(format='jpeg', width=300, height=300)

# Add data to the image widget, data comes from the camera
# Callback function for each time a new frame is collected
def update_image_2(change):
    # Here, "image" is the latest image from the camera
    image = change['new']
    # Process the frame and display it
    #in this area of the callback function, we can draw our line
    # HOW TO USE CV: cv2.line(image_to_change, (x1,y1), (x2,y2), (r,b,g), thickness)
    cv2.line(image, (0,40), (224, 40), (0, 0, 255), 2)
    cv2.line(image, (0,180), (224, 180), (0, 0, 255), 2) 
    image_widget_2.value = bgr8_to_jpeg(image)

# Start the update event observation and display the GUI
                #function: update_image_2
camera.observe(update_image_2, names='value')

display(image_widget_2)

### Next, we need to create two directories where we can save our data.
### Python provides a useful library called "os" that enables us to create directories automatically.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# Run this cell to create two folders to hold your data
blocked_dir = 'dataset/blocked'
free_dir = 'dataset/free'

# we have this "try/except" statement because these next functions can throw an error if the directories exist already
try:
    os.makedirs(free_dir)
    os.makedirs(blocked_dir)
except FileExistsError:
    print('Directories not created because they already exist')

### If you look in your file explorer pane to the left you might see a new directory called "dataset" inside that directory there are two new directories "free" and "blocked"

## Learning How to Create a User Interface
### User interfaces are very important,they need to be easy to understand, and easy to use.
### In the next few cells we're going to explore the process of creating a user interface.
### The process is as follows:
### 1. Create the interface object: buttons, displays, indicators
### 2. Connect those objects with functions, code, or data like the data from the camera
### 3. Arrange the relative position of the objects and finally display them.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# Creating the button widgets for our controller interface, right now they don't do anything =(
# Layout = format or style of the button
button_layout = widgets.Layout(width='100px', height='80px', align_self='center')
collect_button_layout = widgets.Layout(width='128px', height='64px', align_self='center')

# Creating the button objects or "widgets"
stop_button = widgets.Button(description='stop', button_style='danger', layout=button_layout)
forward_button = widgets.Button(description='forward', layout=button_layout)
backward_button = widgets.Button(description='backward', layout=button_layout)
left_button = widgets.Button(description='left', layout=button_layout)
right_button = widgets.Button(description='right', layout=button_layout)

# TWO BUTTONS: "add free" & "add blocked"
free_button = widgets.Button(description='add free', button_style='success',layout=collect_button_layout)
blocked_button = widgets.Button(description='add blocked', button_style='danger', layout=collect_button_layout)

# Creating two displays to show the number of photos in the "free" and "blocked" directories
free_count = widgets.IntText(layout=collect_button_layout, value=len(os.listdir(free_dir)))
blocked_count = widgets.IntText(layout=collect_button_layout, value=len(os.listdir(blocked_dir)))


# Arranging the control buttons and displaying them (#3 on the process)
middle_box = widgets.HBox([left_button,image_widget_2, right_button], layout=widgets.Layout(align_self='center'))
collect_button_boxes = widgets.HBox([free_button, blocked_button], layout=widgets.Layout(align_self='center'))
count_boxes = widgets.HBox([free_count, blocked_count], layout=widgets.Layout(align_self='center'))
controls_box = widgets.VBox([forward_button, middle_box, backward_button,collect_button_boxes, count_boxes, stop_button])
display(controls_box)

### So far, we've created our objects and arranged them into a nice graphical user interface (GUI).
### We next need to create functions for each button and connect thos functions to each button.
### To do this we'll use some of the skills we learned yesterday about raw motor control of the robot.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# Below are the functions we used to move our robot in the last exercise, they should look familiar!

def stop(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.off.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.off.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )
        
    
def move_forward(change):
    print("fire")
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.5)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

def move_backward(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.reverse.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.reverse.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.25)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

def turn_left(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.reverse.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.25)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.off.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.off.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

def turn_right(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.reverse.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
    )
    time.sleep(.25)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.off.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.off.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# My custom functions to collect images from the camera, these functions will be attached to the "add free" and "add blocked" buttons
# New Function: save_snapshot(directory) - directory argument is the location where the snapshot should be saved
# 1. capture photo from camera
# 2. save to either "free" or "blocked" directory
def save_snapshot(directory): # A FUNCTION WE CAN USE ANY TIME
    # 1. Capture photo from camera
    new_photo = image_widget.value # jpeg format

    # 2. save to either "free" or "blocked" directory using OS library
    unique_name = os.path.join(directory, str(uuid1()) + '.jpg')
    # Save file
    with open(unique_name, 'wb') as new_file:
        new_file.write(new_photo)

# Callback Functions
# 1. Free Button Callback function
def free_callback():
    save_snapshot(free_dir) # Collect image, save image to "free" directory
    free_count.value = len(os.listdir(free_dir))

# 2. Blocked Button Callback function
def blocked_callback():
    save_snapshot(blocked_dir) # Collect image, save image to "blocked" directory
    blocked_count.value = len(os.listdir(blocked_dir))

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# Unregister any previously registered handlers to prevent duplicates on re-run
stop_button._click_handlers.callbacks.clear()
forward_button._click_handlers.callbacks.clear()
backward_button._click_handlers.callbacks.clear()
left_button._click_handlers.callbacks.clear()
right_button._click_handlers.callbacks.clear()
free_button._click_handlers.callbacks.clear()
blocked_button._click_handlers.callbacks.clear()

# STEP 2a. Link buttons to actions or callback functions
stop_button.on_click(stop)
forward_button.on_click(move_forward)
backward_button.on_click(move_backward)
left_button.on_click(turn_left)
right_button.on_click(turn_right)

free_button.on_click(lambda x: free_callback())
blocked_button.on_click(lambda x: blocked_callback())
display(controls_box)

## At this point all the buttons on your graphical user interface should work. Go ahead and try to move your robot.

## Your goal is to now move your robot and collect images that depict free and blocked situations. Use chairs, backpacks, shoes, or the wall to block your robot and take photos for your dataset.

## When you've collected **200** images of each class move on to the next notebook ML2 - Training Network.

Answers below: